# Data vs prediction — the charge & time likelihood

Compare a **real, sampled PhotonSim event** against the differentiable **prediction** using
the **likelihood the reconstruction optimizes**: per-PMT Poisson charge NLL + first-arrival
time NLL (`lucid.losses.counts_loss` + `first_arrival_window_nll`).

We use a **tilted** muon (ring on the barrel) and a **stochastic** data event (real Poisson +
TTS), look at the per-PMT predicted-vs-observed charge and first-arrival time, and summarize
with the **per-sensor likelihood distribution and its expected value**.

**Predicted first-arrival** per sensor = the *expected value of the first-arrival order
statistic* over that sensor's photons — each photon weighted by its probability of being
the first to fire (see the computation cell below).

> **Prerequisite:** the example ROOT file comes from `./scripts/download_data.sh` (run it from the repo root first).


In [ ]:
import sys; sys.path.append('..')
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
from jax.scipy.special import gammaln
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import load_detector_params
from lucid.sources.event_io import read_photon_data_from_photonsim, pad_photon_data
from lucid.fitting import track_from_vec9, vec9_from_track
from lucid.losses import first_arrival_window_nll
from lucid.visualization import create_detector_display

GEOM, PHYS = '../config/SK_like_geom_config.json', '../config/SK_like_physics_config.json'
ROOT = '../data/water/muon/1000MeV_100events.root'
K, NBUF, NPH, TTS = 8, 400_000, 250_000, 2.5
GRID = dict(n_cap=100, n_angular=150, n_height=100)
EV, E_TRUTH = 7, 1000.

det = generate_detector(GEOM); ND = len(det.all_points)
dp = load_detector_params(PHYS, num_sensors=ND)
dp = dp._replace(response=dp.response._replace(tts=jnp.asarray(TTS)))

data_sim = setup_event_simulator(GEOM, NBUF, temperature=None, K=K, is_data=True, hit_mode='realistic',
                                 physics_config=PHYS, default_detector_params=dp, particle='muon',
                                 wavelength_mode=True, charge_resolution=None, **GRID)
CHER_BAND = (274.91, 673.83)   # PhotonSim Cherenkov emission band — keeps model/data charge normalization consistent
pred = setup_event_simulator(GEOM, NPH, temperature=0.1, K=K, hit_mode='per_photon', physics_config=PHYS,
                             default_detector_params=True, particle='muon', wavelength_mode=True,
                             pos_grad_threshold=K, n_grad_iters=K, cherenkov_emission_band=CHER_BAND, **GRID)
print(f'{ND} PMTs — ready')

## 1. A realistic, tilted data event

In [ ]:
def rotax(u, deg):
    a = np.radians(deg); ca, sa = np.cos(a), np.sin(a); u = u / np.linalg.norm(u)
    ux = np.array([[0,-u[2],u[1]],[u[2],0,-u[0]],[-u[1],u[0],0]])
    return np.eye(3)*ca + sa*ux + (1-ca)*np.outer(u, u)

R = rotax(np.array([0.3, 0.9, 0.]), 55.); sh = np.array([3.0, -4.0, 2.0]) * 100.   # cm
raw = read_photon_data_from_photonsim(ROOT, EV)
O = np.asarray(raw['photon_origins']).astype(float); ctr = O.mean(0)
raw = dict(raw)
raw['photon_origins'] = (O - ctr) @ R.T + ctr + sh
raw['photon_directions'] = np.asarray(raw['photon_directions']).astype(float) @ R.T
vtx_true = ((np.zeros(3) - ctr) @ R.T + ctr + sh) / 100.0
dir_true = np.array([0., 0., 1.]) @ R.T

pd, _ = pad_photon_data(raw, NBUF)
dummy = track_from_vec9(jnp.array([E_TRUTH, 0,0,0, 0.,1., 0.,1., 0.]))
oc, ot = jax.lax.stop_gradient(data_sim(dummy, jax.random.PRNGKey(7000 + EV), pd))
oc, ot = np.asarray(oc), np.asarray(ot)
print(f'truth: vtx={np.round(vtx_true,2)} m  dir={np.round(dir_true,3)}')
print(f'observed: {int((oc>0).sum())} PMTs lit, total charge {oc.sum():.0f} pe')
create_detector_display(GEOM, sparse=False)(oc, ot, file_name=None, plot_time=False, perc_min=0., perc_max=99.5)

## 2. Prediction + the per-sensor likelihood

Run the per-photon prediction at the truth track. Per sensor we form:
- predicted charge `μ` (sum of photon weights) and the **expected first-arrival** (each photon weighted by its probability of firing first);
- **charge NLL** `μ − n·log μ + log n!` and **time NLL** (`first_arrival_window_nll`).

Then we report the **expected (mean) per-sensor likelihood** over lit sensors.

In [ ]:
v = jnp.asarray(vec9_from_track(E_TRUTH, vtx_true, dir_true, t0=0.))
lw, ft, fi, tot = jax.lax.stop_gradient(pred(track_from_vec9(v), jax.random.PRNGKey(11)))
ocf, otf = jnp.asarray(oc), jnp.asarray(ot)
mu = np.maximum(np.asarray(tot), 1e-8)                     # predicted charge per sensor

# predicted FIRST-ARRIVAL per sensor = EXPECTED value of the first-arrival order statistic.
# Detections are a Poisson process, so photon i is the FIRST to fire with prob p_i ∝ w_i·exp(−C_i),
# where C_i = predicted weight arriving BEFORE t_i at that sensor (survival that nothing fired
# earlier). The predicted first-arrival is E[t] = Σ t_i p_i / Σ p_i (mode = argmax_i p_i).
vmask = np.asarray(lw) > -20.0
fis = np.asarray(fi)[vmask]; fts = np.asarray(ft)[vmask]; ws = np.exp(np.clip(np.asarray(lw)[vmask], -60, 20))
order = np.lexsort((fts, fis)); fis, fts, ws = fis[order], fts[order], ws[order]
excl = np.cumsum(ws) - ws                              # global exclusive prefix weight
bnd = np.ones(len(fis), bool); bnd[1:] = fis[1:] != fis[:-1]
C = excl - excl[bnd][np.cumsum(bnd) - 1]               # weight before photon i WITHIN its sensor
p = ws * np.exp(-C)
num = np.bincount(fis, weights=fts * p, minlength=ND); den = np.bincount(fis, weights=p, minlength=ND)
t_pred = num / np.maximum(den, 1e-12)

# per-sensor likelihood terms
cnll = mu - oc * np.log(mu) + np.asarray(gammaln(jnp.asarray(oc) + 1.0))         # charge NLL / sensor
tnll = np.asarray(first_arrival_window_nll(lw, ft, fi, otf, tot, ocf, ND, sigma=TTS, delta=1.0))  # time NLL / sensor
lit = oc > 0
print(f'expected per-sensor charge NLL : {cnll[lit].mean():7.3f}')
print(f'expected per-sensor time   NLL : {tnll[lit].mean():7.3f}')
print(f'expected per-sensor TOTAL  NLL : {(cnll+tnll)[lit].mean():7.3f}   (over {int(lit.sum())} lit sensors)')

## 3. Per-PMT predicted vs observed (linear)

Each point is a lit PMT. Charge scatters around the diagonal with Poisson spread; first-arrival
clusters on the diagonal within the TTS resolution.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
hi = np.percentile(oc[lit], 99.5)
ax[0].plot([0, hi], [0, hi], 'k--', lw=1, label='perfect')
ax[0].scatter(mu[lit], oc[lit], s=8, alpha=.25, color='#3477b8')
ax[0].set(xlim=(0, hi), ylim=(0, hi), xlabel='predicted charge μ (pe)', ylabel='observed charge n (pe)',
          title=f'Charge (mean NLL/sensor = {cnll[lit].mean():.2f})'); ax[0].legend(); ax[0].grid(alpha=.3)
tm = lit & (t_pred > 0) & (ot > 0)
tlo, thi = np.percentile(ot[tm], 1), np.percentile(ot[tm], 99)
ax[1].plot([tlo, thi], [tlo, thi], 'k--', lw=1, label='perfect')
ax[1].scatter(t_pred[tm], ot[tm], s=8, alpha=.25, color='#c0392b')
ax[1].set(xlim=(tlo, thi), ylim=(tlo, thi), xlabel='predicted first-arrival (ns)',
          ylabel='observed first-arrival (ns)', title=f'Time (mean NLL/sensor = {tnll[lit].mean():.2f})')
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); plt.show()

## 4. Distribution of the per-sensor likelihood

The per-sensor NLL distribution and its **expected value** (mean) summarize the data/prediction
agreement in one number per term.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, vals, name, col in [(ax[0], cnll[lit], 'charge', '#3477b8'), (ax[1], tnll[lit], 'time', '#c0392b')]:
    m = float(np.mean(vals)); b = np.linspace(np.percentile(vals, 0.5), np.percentile(vals, 99), 50)
    a.hist(vals, b, color=col, alpha=.6)
    a.axvline(m, color='k', ls='--', lw=1.5, label=f'E[NLL/sensor] = {m:.2f}')
    a.set(xlabel=f'per-sensor {name} NLL', ylabel='sensors', title=f'{name} likelihood per sensor'); a.legend(); a.grid(alpha=.3)
fig.tight_layout(); plt.show()

## Takeaways

- Data and prediction are compared with the **recon likelihood** (Poisson charge NLL +
  first-arrival time NLL), per sensor — not marginal histograms.
- The **predicted first-arrival** is the expected value of the first-arrival order
  statistic per sensor (the same statistic the time NLL is built on) — a
  probability-weighted combination, not a plain minimum or mean of all photons.
- The single-number summary is the **expected per-sensor NLL**; lower = better agreement, and
  it is exactly what `fit_track_multistart` drives down during reconstruction.